# Two-Stage Emotion Detection System

This notebook implements a two-stage emotion detection system:
- **Stage 1**: Detect faces using pre-trained YOLO
- **Stage 2**: Classify emotions from detected face regions

---

## 1. Import Libraries

In [2]:
import torch
from ultralytics import YOLO
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import shutil
from IPython.display import display, Image as IPImage, clear_output
import matplotlib.pyplot as plt

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## 2. TwoStageEmotionDetector Class

In [3]:
class TwoStageEmotionDetector:
    """
    Two-stage emotion detection system:
    Stage 1: Detect faces using pre-trained YOLO
    Stage 2: Classify emotions from detected face regions
    """
    
    def __init__(self, face_model='yolov8n-face.pt', emotion_model=None, device='cuda'):
        """
        Initialize the two-stage detector.
        
        Args:
            face_model: Pre-trained face detection model path
            emotion_model: Trained emotion classifier path (None for training mode)
            device: Computing device ('cuda' or 'cpu')
        """
        self.device = device if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")
        
        # Load face detection model
        print("Loading face detection model...")
        try:
            self.face_detector = YOLO(face_model)
            print(f"✓ Face detector loaded: {face_model}")
        except:
            print(f"⚠ {face_model} not found, falling back to yolov8n.pt")
            print("  For better results, download yolov8n-face.pt from:")
            print("  https://github.com/akanametov/yolov8-face")
            self.face_detector = YOLO('yolov8n.pt')
        
        # Load or initialize emotion classifier
        if emotion_model and Path(emotion_model).exists():
            self.emotion_classifier = YOLO(emotion_model)
            print(f"✓ Emotion classifier loaded: {emotion_model}")
        else:
            self.emotion_classifier = YOLO('yolov8n-cls.pt')
            print("✓ Emotion classifier initialized for training")
    
    def train_emotion_classifier(self, dataset_dir, epochs=20, imgsz=224, batch=96, patience=15):
        """
        Train the emotion classification model.
        """
        print(f"\n{'='*60}")
        print("Training Emotion Classifier")
        print(f"{'='*60}\n")
        
        results = self.emotion_classifier.train(
            data=dataset_dir,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=self.device,
            patience=patience,
            save=True,
            plots=True,
            project='emotion_runs',
            name='emotion_classifier'
        )
        
        print("\n✓ Training completed!")
        
        # Automatically copy best model to root directory
        best_model_path = Path('emotion_runs/emotion_classifier/weights/best.pt')
        root_model_path = Path('best_emotion_model.pt')
        
        if best_model_path.exists():
            shutil.copy2(best_model_path, root_model_path)
            print(f"✓ Best model automatically saved to: {root_model_path.absolute()}")
        
        return results
    
    def validate_emotion_classifier(self, dataset_dir):
        """Validate the trained emotion classifier."""
        print("\nValidating emotion classifier...")
        return self.emotion_classifier.val(data=dataset_dir)
    
    def detect_and_classify(self, image_path, conf_face=0.5, conf_emotion=0.3, 
                           save_output=False, output_path=None):
        """
        Perform two-stage emotion detection on an image.
        
        Returns:
            annotated_img: Image with bounding boxes and labels
            emotions_detected: List of detected emotions with metadata
        """
        # Load image
        if isinstance(image_path, str):
            img = cv2.imread(image_path)
            if img is None:
                print(f"Error: Could not read image {image_path}")
                return None, []
        else:
            img = image_path
        
        # Stage 1: Detect faces
        face_results = self.face_detector(img, conf=conf_face, verbose=False)
        
        # Check if any faces detected
        if (face_results is None or len(face_results) == 0 or 
            face_results[0].boxes is None or len(face_results[0].boxes) == 0):
            return img, []
        
        # Stage 2: Classify emotion for each detected face
        emotions_detected = []
        annotated_img = img.copy()
        
        for box in face_results[0].boxes:
            # Extract face region
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            face_crop = img[y1:y2, x1:x2]
            
            if face_crop.size == 0:
                continue
            
            # Classify emotion
            emotion_results = self.emotion_classifier(face_crop, conf=conf_emotion, verbose=False)
            
            if len(emotion_results) > 0:
                probs = emotion_results[0].probs
                top_class = probs.top1
                confidence = probs.top1conf.item()
                emotion = emotion_results[0].names[top_class]
                
                emotions_detected.append({
                    'emotion': emotion,
                    'confidence': confidence,
                    'bbox': [x1, y1, x2, y2]
                })
                
                # Annotate image
                color = self._get_emotion_color(emotion)
                cv2.rectangle(annotated_img, (x1, y1), (x2, y2), color, 2)
                
                # Draw label with background
                label = f"{emotion}: {confidence:.2%}"
                (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(annotated_img, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
                cv2.putText(annotated_img, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Print results
        print(f"\nDetected {len(emotions_detected)} face(s):")
        for i, result in enumerate(emotions_detected, 1):
            print(f"  Face {i}: {result['emotion']} ({result['confidence']:.2%})")
        
        # Save output if requested
        if save_output and output_path:
            cv2.imwrite(output_path, annotated_img)
            print(f"\n✓ Saved annotated image to: {output_path}")
        
        return annotated_img, emotions_detected
    
    def process_video(self, video_source, output_path=None, conf_face=0.5, conf_emotion=0.3):
        """
        Process video file or webcam stream for emotion detection.
        """
        # Open video source
        if video_source == 'webcam':
            video_source = 0
        
        cap = cv2.VideoCapture(video_source)
        
        if not cap.isOpened():
            print(f"Error: Could not open video source {video_source}")
            return
        
        # Setup video writer if output requested
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        
        print("\nProcessing video... Press 'q' to quit")
        frame_count = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # Process frame
            annotated_frame, emotions = self.detect_and_classify(
                frame, conf_face=conf_face, conf_emotion=conf_emotion
            )
            
            # Always use the frame (annotated or original)
            display_frame = annotated_frame if annotated_frame is not None else frame
            
            # Add frame info overlay
            info_text = f"Frame: {frame_count} | Faces: {len(emotions) if emotions else 0}"
            cv2.putText(display_frame, info_text, (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow('Emotion Detection', display_frame)
            
            # Save frame if recording
            if output_path:
                out.write(display_frame)
            
            # Check for quit command (MUST be after imshow)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        # Cleanup
        cap.release()
        if output_path:
            out.release()
        cv2.destroyAllWindows()
        print(f"\n✓ Video processing completed! Processed {frame_count} frames")
    
    def _get_emotion_color(self, emotion):
        """Map emotions to BGR colors for visualization."""
        colors = {
            'angry': (0, 0, 255),        # Red
            'disgusted': (0, 128, 128),  # Teal
            'fearful': (128, 0, 128),    # Purple
            'happy': (0, 255, 0),        # Green
            'neutral': (128, 128, 128),  # Gray
            'sad': (255, 0, 0),          # Blue
            'surprised': (0, 255, 255),  # Yellow
            'nothing': (200, 200, 200)   # Light gray
        }
        return colors.get(emotion.lower(), (255, 255, 255))
    
    def export_emotion_model(self, format='onnx'):
        """Export emotion classifier to different formats."""
        print(f"Exporting emotion classifier to {format}...")
        self.emotion_classifier.export(format=format)
        
        # Copy exported model to root directory
        if format == 'onnx':
            onnx_path = Path('emotion_runs/emotion_classifier/weights/best.onnx')
            root_onnx_path = Path('best_emotion_model.onnx')
            if onnx_path.exists():
                shutil.copy2(onnx_path, root_onnx_path)
                print(f"✓ ONNX model saved to: {root_onnx_path.absolute()}")
        
        print("✓ Export completed!")


def check_trained_model():
    """Check if a trained emotion model already exists."""
    possible_paths = [
        Path('best_emotion_model.pt'),
        Path('emotion_runs/emotion_classifier/weights/best.pt'),
    ]
    
    for path in possible_paths:
        if path.exists():
            print(f"\n✓ Found trained model: {path.absolute()}")
            return str(path)
    
    return None


print("✓ TwoStageEmotionDetector class defined!")

✓ TwoStageEmotionDetector class defined!


## 3. Check for Existing Model

In [4]:
print("="*60)
print("TWO-STAGE EMOTION DETECTION SYSTEM")
print("="*60 + "\n")

trained_model_path = check_trained_model()

if trained_model_path:
    print("\n" + "="*60)
    print("TRAINED MODEL FOUND - SKIPPING TRAINING")
    print("="*60)
    print(f"\nUsing existing model: {trained_model_path}")
    print("\nTo retrain, delete or rename the model file:")
    print("  - best_emotion_model.pt")
    print("  - emotion_runs/emotion_classifier/weights/best.pt\n")
else:
    print("\n" + "="*60)
    print("NO TRAINED MODEL FOUND")
    print("="*60)
    print("\nYou will need to train a model first (see next section)")

TWO-STAGE EMOTION DETECTION SYSTEM


✓ Found trained model: c:\Users\nicho\Documents\GitHub_Repository\Python\2331182-Lab-AI\uas\best_emotion_model.pt

TRAINED MODEL FOUND - SKIPPING TRAINING

Using existing model: best_emotion_model.pt

To retrain, delete or rename the model file:
  - best_emotion_model.pt
  - emotion_runs/emotion_classifier/weights/best.pt



## 4. Training (Run only if no model exists)

**Note:** Skip this section if you already have a trained model!

In [ ]:
# Initialize detector for training
detector_train = TwoStageEmotionDetector(
    face_model='yolov8n-face.pt',
    device='cuda'
)

# Configure your dataset path
dataset_path = "data"

# Check if dataset exists
if not Path(dataset_path).exists():
    print(f"\n⚠ Error: Dataset directory not found: {dataset_path}")
    print("Please create dataset with structure:")
    print("  data/")
    print("    train/")
    print("      angry/")
    print("      happy/")
    print("      ...")
    print("    test/")
    print("      angry/")
    print("      happy/")
    print("      ...")
else:
    # Train the emotion classifier
    results = detector_train.train_emotion_classifier(
        dataset_path,
        epochs=20,
        batch=96,
        imgsz=224,
        patience=15
    )
    
    # Validate trained model
    detector_train.validate_emotion_classifier(dataset_path)
    
    # Export model
    print("\n" + "="*60)
    print("EXPORTING MODEL")
    print("="*60 + "\n")
    detector_train.export_emotion_model(format='onnx')
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print("\nModel saved to root directory:")
    print("  - best_emotion_model.pt")
    print("  - best_emotion_model.onnx\n")

Using device: cuda
Loading face detection model...
✓ Face detector loaded: yolov8n-face.pt
✓ Emotion classifier initialized for training

Training Emotion Classifier

New https://pypi.org/project/ultralytics/8.3.240 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.232  Python-3.13.7 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=96, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=

## 5. Load Model for Inference

In [ ]:
# Get the trained model path
model_path = check_trained_model()

if model_path is None:
    print("⚠ No trained model found! Please train a model first (Section 4)")
else:
    # Load trained models for inference
    detector = TwoStageEmotionDetector(
        face_model='yolov8n-face.pt',
        emotion_model=model_path,
        device='cuda'
    )
    
    print("\n" + "="*60)
    print("READY FOR INFERENCE!")
    print("="*60 + "\n")


✓ Found trained model: c:\Users\nicho\Documents\GitHub_Repository\Python\2331182-Lab-AI\uas\best_emotion_model.pt
Using device: cuda
Loading face detection model...
✓ Face detector loaded: yolov8n-face.pt
✓ Emotion classifier loaded: best_emotion_model.pt

READY FOR INFERENCE!



## 6. Real-time Webcam Detection

In [ ]:
# Real-time webcam detection
# Press 'q' to quit

detector.process_video(
    'webcam',
    conf_face=0.5,
    conf_emotion=0.3
)


Processing video... Press 'q' to quit

Detected 1 face(s):
  Face 1: angry (55.93%)

Detected 1 face(s):
  Face 1: surprised (34.33%)

Detected 1 face(s):
  Face 1: angry (32.57%)

Detected 1 face(s):
  Face 1: angry (35.62%)

Detected 1 face(s):
  Face 1: angry (35.62%)

Detected 1 face(s):
  Face 1: angry (36.98%)

Detected 1 face(s):
  Face 1: angry (36.98%)

Detected 1 face(s):
  Face 1: angry (41.87%)

Detected 1 face(s):
  Face 1: angry (41.87%)

Detected 1 face(s):
  Face 1: surprised (30.92%)

Detected 1 face(s):
  Face 1: surprised (30.92%)

Detected 1 face(s):
  Face 1: angry (34.78%)

Detected 1 face(s):
  Face 1: angry (34.78%)

Detected 1 face(s):
  Face 1: surprised (33.89%)

Detected 1 face(s):
  Face 1: surprised (33.89%)

Detected 1 face(s):
  Face 1: angry (40.76%)

Detected 1 face(s):
  Face 1: angry (40.76%)

Detected 1 face(s):
  Face 1: angry (37.89%)

Detected 1 face(s):
  Face 1: angry (37.89%)

Detected 1 face(s):
  Face 1: angry (41.03%)

Detected 1 face(s):


## 7. Helper: Display Training Metrics

In [1]:
# Display training results (if available)
results_path = Path('emotion_runs/emotion_classifier')

if results_path.exists():
    # Display confusion matrix
    confusion_matrix = results_path / 'confusion_matrix_normalized.png'
    if confusion_matrix.exists():
        print("Confusion Matrix:")
        display(IPImage(filename=str(confusion_matrix)))
    
    # Display training curves
    results_png = results_path / 'results.png'
    if results_png.exists():
        print("\nTraining Results:")
        display(IPImage(filename=str(results_png)))
else:
    print("⚠ No training results found. Train a model first!")

NameError: name 'Path' is not defined

# 8. Confusion Matrix

In [ ]:
# ===== CONFUSION MATRIX & ACCURACY EVALUATION (ONE CELL) =====

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score

# CONFIG
DATASET_DIR = "data"
SPLIT = "test"   # use test set only (IMPORTANT)
MODEL = detector.emotion_classifier  # trained YOLOv8-CLS model

# Load class names from folder structure
dataset_path = Path(DATASET_DIR) / SPLIT
class_names = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])

y_true = []
y_pred = []

print("Evaluating emotion classifier...\n")

# Iterate through dataset
for label_idx, class_name in enumerate(class_names):
    class_dir = dataset_path / class_name
    
    for img_path in class_dir.glob("*.*"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        results = MODEL(img, verbose=False)
        probs = results[0].probs
        
        pred_class = probs.top1
        y_pred.append(pred_class)
        y_true.append(label_idx)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {accuracy:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Emotion Classification")
plt.tight_layout()
plt.show()


## Notes

**Dataset Structure:**
```
data/
├── train/
│   ├── angry/
│   ├── happy/
│   ├── sad/
│   ├── surprised/
│   ├── neutral/
│   ├── fearful/
│   └── disgusted/
└── test/
├── angry/
├── happy/
├── sad/
└── 
```

**Trained Models Location:**
- `best_emotion_model.pt` → PyTorch model (di root directory)
- `best_emotion_model.onnx` → ONNX format (di root directory)
- `emotion_runs/emotion_classifier/weights/` → Semua output training lengkap

**Tips:**
- Sesuaikan `conf_face` dan `conf_emotion` sesuai kebutuhan
- Nilai lebih rendah → lebih banyak deteksi (tapi risiko false positive lebih tinggi)
- Nilai lebih tinggi → lebih sedikit tapi lebih yakin
- Gunakan GPU (`device='cuda'`) untuk proses lebih cepat
- Tekan tombol **q** untuk menghentikan proses video/webcam

**Common Issues & Solusi:**
- Webcam tidak jalan → coba ubah `video_source` jadi 1, 2, dst (tergantung kamera eksternal)
- Tidak ada wajah terdeteksi → turunkan `conf_face` (misal jadi 0.3 atau 0.4)
- Hasil emosi sering salah → pertimbangkan retraining dengan data lebih banyak/lebih baik
- Face detection kurang akurat → gunakan model khusus face seperti `yolov8s-face.pt` atau `yolov8n-face.pt` (bukan fallback ke `yolov8n.pt`)

**Model Export Formats yang Didukung:**
- **ONNX** → Cross-platform deployment
- **TorchScript** → Produksi PyTorch
- **CoreML** → Aplikasi iOS/macOS
- **TensorFlow Lite** → Mobile & embedded devices